# Dueling DQN Training - IMC Prosperity 4

**What this notebook does:**
1. Installs dependencies (stable-baselines3, gymnasium)
2. Uploads/mounts your trading data
3. Computes 19 trading indicators from order book data
4. Trains a Dueling DQN agent to trade EMERALDS & TOMATOES
5. Evaluates on held-out data and exports weights for submission

**Run this on:** Google Colab (free GPU) or Kaggle Notebooks

---

## 1. Setup & Install Dependencies

In [ ]:
!pip install stable-baselines3 gymnasium pandas numpy matplotlib --quiet
print("Dependencies installed!")

## 2. Setup Paths

In [ ]:
import os, sys

# Notebook is at RLM/dueling_dqn/train.ipynb => repo root is ../..
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

## 3. Load Data & Check GPU

In [ ]:
import torch
import numpy as np
import pandas as pd
import os

# Check GPU availability
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("No GPU detected - training on CPU (still fast for this model)")

# Load data
from RLM.shared.data_loader import load_prices, load_trades

prices_df = load_prices(data_dir=DATA_DIR)
trades_df = load_trades(data_dir=DATA_DIR)

print(f"\nPrices: {len(prices_df)} rows")
print(f"Trades: {len(trades_df)} rows")
print(f"Products: {prices_df['product'].unique()}")
print(f"Days: {sorted(prices_df['day'].unique())}")

## 4. Compute Feature Normalization

Before training, we need to compute the mean and standard deviation of each feature across the training data. This ensures all 19 features are on the same scale.

In [ ]:
from RLM.shared.config import PRODUCTS, TRAIN_CONFIG, DQN_CONFIG, NETWORK_CONFIG, ENV_CONFIG
from RLM.shared.features import FeatureComputer, compute_features_from_row, fit_normalizer
from RLM.shared.env import TradingEnv

# Compute normalization parameters from training data
def compute_normalization_params(prices_df, trades_df, products, days):
    all_features = {p: [] for p in products}
    for day in days:
        for product in products:
            fc = FeatureComputer(product)
            day_prices = prices_df[(prices_df["day"] == day) & (prices_df["product"] == product)]
            day_prices = day_prices.sort_values("timestamp").reset_index(drop=True)
            day_trades = trades_df[trades_df["symbol"] == product].sort_values("timestamp")
            for _, row in day_prices.iterrows():
                ts = row["timestamp"]
                ts_trades = day_trades[day_trades["timestamp"] == ts]
                trades = list(zip(ts_trades["price"], ts_trades["quantity"])) if len(ts_trades) > 0 else None
                features = compute_features_from_row(row, fc, position=0, trades=trades)
                all_features[product].append(features)
    combined = np.vstack([np.array(v) for v in all_features.values()])
    return fit_normalizer(combined)

feat_means, feat_stds = compute_normalization_params(
    prices_df, trades_df, PRODUCTS, TRAIN_CONFIG["train_days"]
)

print("Feature normalization computed!")
print(f"Means (first 5): {feat_means[:5]}")
print(f"Stds  (first 5): {feat_stds[:5]}")

## 5. Create Environments & Train Dueling DQN

This creates the training env (with data augmentation) and eval env (clean data), then trains the Dueling DQN agent. The network splits into Value V(s) and Advantage A(s,a) streams:

    Q(s,a) = V(s) + A(s,a) - mean(A(s,:))

**Adjust `TOTAL_TIMESTEPS` below** - more steps = better policy but longer training.

In [ ]:
# ============================================================
# TRAINING CONFIG - Adjust these!
# ============================================================
TOTAL_TIMESTEPS = 100_000   # Increase for better results (200k, 500k)
SEED = 42
LEARNING_RATE = 1e-3        # Set to None to use default from config
# ============================================================

import torch
import torch.nn as nn
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback
from RLM.dueling_dqn.train import DuelingQNetwork
from RLM.shared.config import NUM_FEATURES, NUM_ACTIONS

# Create training environment (with data augmentation)
train_env = TradingEnv(
    prices_df=prices_df, trades_df=trades_df,
    products=PRODUCTS, day=TRAIN_CONFIG["train_days"][0],
    augment=True, seed=SEED,
)

# Create eval environment (no augmentation)
eval_env = TradingEnv(
    prices_df=prices_df, trades_df=trades_df,
    products=PRODUCTS, day=TRAIN_CONFIG["eval_days"][0],
    augment=False, seed=SEED + 1,
)

# Set normalization params
for product in PRODUCTS:
    train_env.feature_computers[product].feature_means = feat_means
    train_env.feature_computers[product].feature_stds = feat_stds
    eval_env.feature_computers[product].feature_means = feat_means
    eval_env.feature_computers[product].feature_stds = feat_stds

# Output directory
MODEL_DIR = os.path.join(PROJECT_ROOT, "RLM", "dueling_dqn", "policy_weights")
os.makedirs(MODEL_DIR, exist_ok=True)

# Eval callback - saves best model during training
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=MODEL_DIR,
    log_path=MODEL_DIR,
    eval_freq=max(TOTAL_TIMESTEPS // 20, 1000),
    n_eval_episodes=5,
    deterministic=True,
    verbose=1,
)

# Dimensions
obs_dim = NUM_FEATURES * len(PRODUCTS)
action_dim = NUM_ACTIONS ** len(PRODUCTS)

# Create DQN model with MlpPolicy as base
model = DQN(
    "MlpPolicy",
    train_env,
    learning_rate=LEARNING_RATE or DQN_CONFIG["learning_rate"],
    buffer_size=DQN_CONFIG["buffer_size"],
    learning_starts=DQN_CONFIG["learning_starts"],
    batch_size=DQN_CONFIG["batch_size"],
    gamma=DQN_CONFIG["gamma"],
    exploration_fraction=DQN_CONFIG["exploration_fraction"],
    exploration_initial_eps=DQN_CONFIG["exploration_initial_eps"],
    exploration_final_eps=DQN_CONFIG["exploration_final_eps"],
    target_update_interval=DQN_CONFIG["target_update_interval"],
    train_freq=DQN_CONFIG["train_freq"],
    gradient_steps=DQN_CONFIG["gradient_steps"],
    policy_kwargs=dict(net_arch=NETWORK_CONFIG["hidden_sizes"]),
    device="auto",
    verbose=1,
    seed=SEED,
)

# Replace q_net with Dueling architecture
dueling_net = DuelingQNetwork(obs_dim, action_dim, NETWORK_CONFIG["hidden_sizes"])
model.q_net.q_net = dueling_net.to(model.device)
model.q_net_target.q_net = DuelingQNetwork(obs_dim, action_dim, NETWORK_CONFIG["hidden_sizes"]).to(model.device)
# Copy weights to target
model.q_net_target.load_state_dict(model.q_net.state_dict())

# Rebuild optimizer with new parameters
model.policy.optimizer = torch.optim.Adam(
    model.q_net.parameters(),
    lr=LEARNING_RATE or DQN_CONFIG["learning_rate"],
)

print(f"Device: {model.device}")
print(f"Architecture: Dueling (Value + Advantage streams)")
print(f"Hidden sizes: {NETWORK_CONFIG['hidden_sizes']}")
print(f"Training for {TOTAL_TIMESTEPS:,} timesteps...")
print()

# TRAIN!
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=eval_callback,
    progress_bar=True,
)

# Save final model + normalization params
model.save(os.path.join(MODEL_DIR, "final_model"))
np.savez(os.path.join(MODEL_DIR, "norm_params.npz"),
         feature_means=feat_means, feature_stds=feat_stds)

print("\nTraining complete!")

## 6. Evaluate on Held-Out Day

Run the trained agent on the eval day (which it never saw during training) and check PnL.

In [ ]:
# Evaluate on held-out data
import matplotlib.pyplot as plt

n_eval_episodes = 10
episode_pnls = []
episode_rewards = []

for ep in range(n_eval_episodes):
    obs, info = eval_env.reset()
    total_reward = 0
    done = False
    pnl_curve = [0]

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        pnl_curve.append(info.get("pnl", total_reward))
        done = terminated or truncated

    episode_pnls.append(info.get("pnl", total_reward))
    episode_rewards.append(total_reward)
    print(f"Episode {ep+1}: PnL={episode_pnls[-1]:.2f}, Positions={info.get('positions', {})}")

pnls = np.array(episode_pnls)
print(f"\n{'='*40}")
print(f"Mean PnL:  {pnls.mean():.2f}")
print(f"Std PnL:   {pnls.std():.2f}")
print(f"Min PnL:   {pnls.min():.2f}")
print(f"Max PnL:   {pnls.max():.2f}")
if pnls.std() > 0:
    print(f"Sharpe:    {pnls.mean() / pnls.std():.2f}")

## 7. Export Weights to Numpy

Extract the trained Dueling DQN weights into a `.npz` file. Uses stream-specific naming (shared_W0, value_W0, advantage_W0) for the NumpyDuelingMLP inference class.

In [ ]:
# Export trained Dueling DQN weights to numpy .npz
weights = {}

# Access the DuelingQNetwork inside the SB3 model
dueling = model.q_net.q_net

# Shared layers
shared_idx = 0
for name, param in dueling.shared.named_parameters():
    tensor = param.detach().cpu().numpy()
    print(f"  shared.{name}: {tensor.shape}")
    if "weight" in name:
        weights[f"shared_W{shared_idx}"] = tensor
    elif "bias" in name:
        weights[f"shared_B{shared_idx}"] = tensor
        shared_idx += 1

# Value stream
val_idx = 0
for name, param in dueling.value_stream.named_parameters():
    tensor = param.detach().cpu().numpy()
    print(f"  value_stream.{name}: {tensor.shape}")
    if "weight" in name:
        weights[f"value_W{val_idx}"] = tensor
    elif "bias" in name:
        weights[f"value_B{val_idx}"] = tensor
        val_idx += 1

# Advantage stream
adv_idx = 0
for name, param in dueling.advantage_stream.named_parameters():
    tensor = param.detach().cpu().numpy()
    print(f"  advantage_stream.{name}: {tensor.shape}")
    if "weight" in name:
        weights[f"advantage_W{adv_idx}"] = tensor
    elif "bias" in name:
        weights[f"advantage_B{adv_idx}"] = tensor
        adv_idx += 1

# Include normalization params
weights["feature_means"] = feat_means
weights["feature_stds"] = feat_stds

# Save
export_path = os.path.join(MODEL_DIR, "exported_weights.npz")
np.savez(export_path, **weights)

total_params = sum(v.size for k, v in weights.items() if "feature" not in k)
file_size = os.path.getsize(export_path)
print(f"\nExported {len(weights)} arrays, {total_params:,} parameters")
print(f"File size: {file_size / 1024:.1f} KB")
print(f"Saved to: {export_path}")

# Verify with numpy forward pass
from RLM.shared.numpy_policy import NumpyDuelingMLP
numpy_model = NumpyDuelingMLP(weights_path=export_path)
test_obs = np.random.randn(NUM_FEATURES * len(PRODUCTS)).astype(np.float32)
action, q_vals = numpy_model.predict(test_obs, normalize=False)
print(f"\nVerification pass: action={action}, Q-values={q_vals[:3]}...")

## 8. Download Weights

Run this cell to download the exported weights to your local machine. Then use `build_submission.py` locally to create the submission file.

In [ ]:
# Download/save weights
try:
    from google.colab import files
    files.download(export_path)
    print("Download started! Check your browser downloads.")
except ImportError:
    print("Not running on Colab. Weights saved to:", export_path)
except Exception as e:
    print(f"Colab download not available ({e}).")
    print(f"Weights saved to: {export_path}")
    print("Access them from the file browser or copy from the path above.")